# RGB-Agent Local Evaluation

This notebook is for running `RGB-Agent` locally on your own machine.

It supports:
- installing this project in editable mode
- optionally starting a local `vLLM` OpenAI-compatible server
- running `RGB-Agent` offline evaluation against local `environment_files/`
- inspecting `summary.txt`, `summary.csv`, and `scorecard.json`

Recommended local setup for your machine:
- GPU: `RTX PRO 6000 Blackwell 96GB`
- Model: `Qwen/Qwen2.5-72B-Instruct-AWQ`
- Backend: `vLLM`
- Analyzer backend: `direct`


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists() or not (PROJECT_ROOT / 'rgb_agent').exists():
    raise FileNotFoundError('Run this notebook from the RGB-Agent project root.')

ENV_DIR = PROJECT_ROOT / 'environment_files'
OUTPUT_ROOT = PROJECT_ROOT / 'evaluation_results'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('ENV_DIR      =', ENV_DIR)
print('OUTPUT_ROOT  =', OUTPUT_ROOT)


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)])
print('Project installed in editable mode.')


## Configuration

Edit this cell before running.

Typical local use:
- point `MODEL_DIR` at your downloaded model directory
- leave `START_VLLM=True`
- use the `AWQ` model variant for single-GPU serving


In [ ]:
AGENT_NAME = 'rgb_agent'
SUITE = 'all'
GAME = None
MAX_ACTIONS = 500
ANALYZER_INTERVAL = 10
ANALYZER_RETRIES = 5
DESCRIPTION = 'local-qwen72b-awq'

START_VLLM = True
MODEL_DIR = Path('/absolute/path/to/Qwen2.5-72B-Instruct-AWQ')
SERVED_MODEL_NAME = 'Qwen/Qwen2.5-72B-Instruct-AWQ'

ANALYZER_BACKEND = 'direct'
MODEL = f'openai/{SERVED_MODEL_NAME}'

VLLM_HOST = '127.0.0.1'
VLLM_PORT = 8000
VLLM_GPU_MEMORY_UTILIZATION = 0.92
VLLM_MAX_MODEL_LEN = 16384
VLLM_MAX_NUM_SEQS = 8
VLLM_TENSOR_PARALLEL_SIZE = 1

OPENAI_BASE_URL = f'http://{VLLM_HOST}:{VLLM_PORT}/v1'
OPENAI_API_KEY = 'EMPTY'

os.environ['ENVIRONMENTS_DIR'] = str(ENV_DIR)
os.environ['ANALYZER_BACKEND'] = ANALYZER_BACKEND
os.environ['OPENAI_BASE_URL'] = OPENAI_BASE_URL
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print('MODEL_DIR        =', MODEL_DIR)
print('MODEL            =', MODEL)
print('OPENAI_BASE_URL  =', OPENAI_BASE_URL)


In [ ]:
import requests
import subprocess
import time

vllm_proc = None

if START_VLLM:
    if not MODEL_DIR.exists():
        raise FileNotFoundError(f'Model directory not found: {MODEL_DIR}')

    cmd = [
        'vllm',
        'serve',
        str(MODEL_DIR),
        '--served-model-name', SERVED_MODEL_NAME,
        '--host', VLLM_HOST,
        '--port', str(VLLM_PORT),
        '--gpu-memory-utilization', str(VLLM_GPU_MEMORY_UTILIZATION),
        '--max-model-len', str(VLLM_MAX_MODEL_LEN),
        '--max-num-seqs', str(VLLM_MAX_NUM_SEQS),
        '--tensor-parallel-size', str(VLLM_TENSOR_PARALLEL_SIZE),
        '--generation-config', 'vllm',
        '--trust-remote-code',
    ]
    print('Starting vLLM:')
    print(' '.join(cmd))
    vllm_proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

    ready = False
    for _ in range(180):
        try:
            r = requests.get(f'http://{VLLM_HOST}:{VLLM_PORT}/v1/models', timeout=1)
            if r.ok:
                ready = True
                break
        except Exception:
            time.sleep(1)

    if not ready:
        raise RuntimeError('vLLM server did not become ready in time.')
    print('vLLM is ready.')
else:
    print('Skipping vLLM startup.')


In [ ]:
from rgb_agent.evaluation.offline_eval import run_offline_evaluation

result = run_offline_evaluation(
    agent_name=AGENT_NAME,
    game=GAME,
    suite=SUITE,
    max_actions=MAX_ACTIONS,
    analyzer_interval=ANALYZER_INTERVAL,
    analyzer_model=MODEL,
    analyzer_backend=ANALYZER_BACKEND,
    analyzer_retries=ANALYZER_RETRIES,
    description=DESCRIPTION,
    output_root=OUTPUT_ROOT,
    environments_dir=ENV_DIR,
)

result


In [ ]:
from pathlib import Path
import json

run_dir = Path(result['run_dir'])
print('Run dir:', run_dir)
print('\nArtifacts:')
for path in sorted(run_dir.iterdir()):
    print('-', path.name)

if result.get('scorecard_json'):
    scorecard = json.loads(Path(result['scorecard_json']).read_text())
    print('\nOverall score:', scorecard.get('score'))


In [ ]:
summary_path = Path(result['summary_txt'])
print(summary_path.read_text())


In [ ]:
import pandas as pd

csv_path = Path(result['summary_csv'])
df = pd.read_csv(csv_path)
df
